Causal test, stage 1 (patch-h readout). On the cluster, `scripts/causal_readout.py` pushed the English harm direction `h` onto every instruction token at `h_layer` and recorded where `h` and the refusal signal `r` ended up, for the harmful train prompts and harmless val. Here we join that with what the model actually did on each prompt (refused or complied, notebook 09) and ask: if a complier gets an English-sized harm signal, does its refusal signal climb to where the refusers are? Forward passes only: this reads `r`, it does not generate text yet.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from harm_route.judge import is_garbled, to_class

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
name = "qwen3-8b"
judge = "qwen3guard-4b"
langs = ["eng_Latn", "npi_Deva", "srp_Cyrl", "srp_Latn"]
conds = ["base", "topup_h", "topup_rand", "fixed_h", "fixed_rand"]
script = {l: v["script"] for l, v in yaml.safe_load(open(ROOT / "configs/languages.yaml")).items()}

O = {l: torch.load(ROOT / f"results/e01_causal/{name}/{l}_readout.pt") for l in langs}   # [lang][set]["h"|"r"][cond]: [N, L]
dirs = torch.load(ROOT / f"data/cache/e00/{name}_directions.pt")
H_LAYER, R_LAYER = dirs["h_layer"], dirs["r_layer"]
n_layers = dirs["h"].shape[0]
LATE = n_layers - 2                                                   # late readout layer, after every push
meta = O["npi_Deva"]["meta"]
print(name, "h_layer", H_LAYER, "r_layer", R_LAYER, "late", LATE, "| gap", round(meta["gap"], 1), meta["dtype"])

The English scale, as in notebook 09: per layer, 0 = English harmless train mean and 1 = English harmful train mean. Only the English cache is needed for this.

In [ ]:
E = torch.load(ROOT / f"data/cache/e01/{name}/eng_Latn_acts.pt")
H = dirs["h"] / dirs["h"].norm(dim=-1, keepdim=True)                   # [L, d]
R = dirs["r"][-1] / dirs["r"][-1].norm(dim=-1, keepdim=True)           # [L, d]
def levels(key, D):
    lo = torch.einsum("nld,ld->nl", E["harmless_train"][key], D).mean(0)   # [L]
    hi = torch.einsum("nld,ld->nl", E["harmful_train"][key], D).mean(0)    # [L]
    return lo, hi
(h_lo, h_hi), (r_lo, r_hi) = levels("inst", H), levels("last", R)
del E
def sc(x, lo, hi):
    return (x - lo) / (hi - lo)                                        # [N, L] - [L] broadcasts over prompts
S = {l: {st: {k: {c: sc(O[l][st][k][c], *((h_lo, h_hi) if k == "h" else (r_lo, r_hi))) for c in conds}
              for k in ("h", "r")} for st in ("harmful_train", "harmless_val")} for l in langs}

Labels from notebook 09: refused = clean refusal, complied = harmful or benign compliance; refuse-then-leak and garbled are left out.

In [ ]:
def groups(lang):
    rows = [json.loads(x) for x in open(ROOT / f"data/generations/e01_scale/{name}/{lang}/judge/{judge}/harmful_train__base.jsonl")]
    assert [r["idx"] for r in rows] == list(range(len(rows)))
    cls = [to_class(r, is_garbled(r["n_tokens"], r["d4"], r["script"], script[lang])) for r in rows]
    return torch.tensor([1 if c == "refuse" else 0 if c in ("comply_harmful", "comply_benign") else -1 for c in cls])
G = {l: groups(l) for l in langs}
for l in langs:
    assert len(G[l]) == S[l]["harmful_train"]["r"]["base"].shape[0]
    print(f"{l:9s} refused {(G[l] == 1).sum():3d}  complied {(G[l] == 0).sum():3d}")

Sanity: after a top-up, every complier's `h` at `h_layer` should sit at 1 (it was pushed exactly to the English harmful level), the random push should leave `h` where it was, and the base numbers should match notebook 09 (Nepali compliers about 0.13 on h, 0.22 to 0.27 on r).

In [ ]:
for l in langs:
    m = G[l] == 0
    if m.sum() == 0:
        continue
    h = S[l]["harmful_train"]["h"]
    print(f"{l:9s} compliers h@L{H_LAYER}: base {h['base'][m, H_LAYER].mean():.2f}  topup_h {h['topup_h'][m, H_LAYER].mean():.2f}"
          f"  topup_rand {h['topup_rand'][m, H_LAYER].mean():.2f}  | mean push {O[l]['harmful_train']['push']['topup_h'][m].mean():.1f}")

The main table: mean refusal signal `r` (English scale) for each group under each push, at `r_layer` and at a late layer. The question is whether the compliers' row climbs toward the refusers' base value, and whether the random push leaves it alone.

In [ ]:
def row(l, st, mask, layer):
    return [S[l][st]["r"][c][mask, layer].mean().item() for c in conds]

for layer in (R_LAYER, LATE):
    print(f"\nr at L{layer} (0 = Eng harmless, 1 = Eng harmful)")
    print(f"{'lang':9s} {'group':9s} {'n':>3s} " + " ".join(f"{c:>10s}" for c in conds))
    for l in langs:
        for gname, st, mask in [("refused", "harmful_train", G[l] == 1), ("complied", "harmful_train", G[l] == 0),
                                ("harmless", "harmless_val", torch.ones(S[l]["harmless_val"]["r"]["base"].shape[0], dtype=torch.bool))]:
            if mask.sum() == 0:
                continue
            print(f"{l:9s} {gname:9s} {mask.sum():3d} " + " ".join(f"{v:10.2f}" for v in row(l, st, mask, layer)))

One number per language: how much of the gap between compliers and refusers (in `r`, before any push) does topping up `h` close? 1 = compliers now look like refusers, 0 = no change. The random version should be near 0. Bootstrap over compliers for a confidence interval.

In [ ]:
def gap_closed(l, cond, layer, idx=None):
    r = S[l]["harmful_train"]["r"]
    c = (G[l] == 0).nonzero().squeeze(1)                               # complier indices
    c = c if idx is None else c[idx]
    ref = r["base"][G[l] == 1, layer].mean()
    return ((r[cond][c, layer].mean() - r["base"][c, layer].mean()) / (ref - r["base"][c, layer].mean())).item()

g = torch.Generator().manual_seed(0)
print(f"fraction of the complier-refuser r gap closed, at L{R_LAYER} and L{LATE} [95% bootstrap CI]")
for l in langs[1:]:
    n = int((G[l] == 0).sum())
    for layer in (R_LAYER, LATE):
        out = []
        for cond in ("topup_h", "topup_rand", "fixed_h", "fixed_rand"):
            boots = torch.tensor([gap_closed(l, cond, layer, torch.randint(n, (n,), generator=g)) for _ in range(1000)])
            out.append(f"{cond} {gap_closed(l, cond, layer):5.2f} [{boots.quantile(0.025):5.2f}, {boots.quantile(0.975):5.2f}]")
        print(f"{l:9s} L{layer:2d}  " + " | ".join(out))

Per prompt: each complier's `r` before (x) and after (y) the top-up, at `r_layer`. Points above the diagonal went up; the dashed line is the refusers' average.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
for ax, l in zip(axes, langs[1:]):
    r = S[l]["harmful_train"]["r"]; m = G[l] == 0
    ax.scatter(r["base"][m, R_LAYER], r["topup_h"][m, R_LAYER], s=16, label="top-up h")
    ax.scatter(r["base"][m, R_LAYER], r["topup_rand"][m, R_LAYER], s=16, marker="x", label="top-up random")
    ax.axline((0, 0), slope=1, color="k", lw=0.8)
    ax.axhline(r["base"][G[l] == 1, R_LAYER].mean(), color="gray", ls="--", lw=0.8)
    ax.set_title(f"{l}: compliers, r at L{R_LAYER}"); ax.set_xlabel("before"); ax.set_ylabel("after")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

Over depth: the compliers' mean `r` at every layer from the push layer on, base vs top-up vs random, and the refusers for reference. This shows whether the effect of the push grows, holds or fades on the way to the output.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
Ls = torch.arange(H_LAYER, n_layers)
for ax, l in zip(axes, langs[1:]):
    r = S[l]["harmful_train"]["r"]; m = G[l] == 0
    for c, st in [("base", "-"), ("topup_h", "-"), ("topup_rand", ":")]:
        ax.plot(Ls, r[c][m][:, Ls].mean(0), st, label=f"compliers {c}")
    ax.plot(Ls, r["base"][G[l] == 1][:, Ls].mean(0), "k--", lw=0.8, label="refusers base")
    ax.set_title(l); ax.set_xlabel("layer")
axes[0].set_ylabel("r (English scale)"); axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

Layer sweep. Same pushes, but now done at one of several layers (fractions 0.2, 0.3, 0.4, 0.5 of depth, plus `h_layer`), each time with that layer's own English `h` direction and English targets. The question: does pushing harm earlier, before the refusal signal has split refusers from compliers, close more of the gap? Late layers 34-35 jump around for every group, so the late readout here is the average of `r` over the layers from `r_layer` to `n_layers - 3`.

In [ ]:
W = {l: torch.load(ROOT / f"results/e01_causal/{name}/{l}_readout_sweep.pt") for l in langs}
PUSH = W["npi_Deva"]["meta"]["push_layers"]
late = list(range(R_LAYER, n_layers - 2))                             # readout window, e.g. L22..L33 on 8B
def r_read(l, st, key, where):
    r = sc(W[l][st]["r"][key], r_lo, r_hi)                            # [N, L], English scale per layer
    return r[:, R_LAYER] if where == "r_layer" else r[:, late].mean(1)   # [N]
assert torch.allclose(W["npi_Deva"]["harmful_train"]["r"]["base"], O["npi_Deva"]["harmful_train"]["r"]["base"])  # same base as stage 1
print("push layers", PUSH, "| late window", late[0], "-", late[-1])

Fraction of the complier-to-refuser gap closed, for each push layer. Solid = push along `h`, dotted = same push along a random direction. Left: top-up (each prompt raised to the English harmful level at that layer). Right: fixed push (the same size for everyone).

In [ ]:
def closed(l, key, where):
    c, rf = G[l] == 0, G[l] == 1
    base = r_read(l, "harmful_train", "base", where)
    after = r_read(l, "harmful_train", key, where)
    return ((after[c].mean() - base[c].mean()) / (base[rf].mean() - base[c].mean())).item()

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for row, where in enumerate(["r_layer", "late"]):
    for col, kind in enumerate(["topup", "fixed"]):
        ax = axes[row, col]
        for i, l in enumerate(langs[1:]):
            ax.plot(PUSH, [closed(l, f"{kind}_h@L{L}", where) for L in PUSH], "-o", color=f"C{i}", label=l)
            ax.plot(PUSH, [closed(l, f"{kind}_rand@L{L}", where) for L in PUSH], ":", color=f"C{i}")
        ax.axhline(0, color="k", lw=0.6); ax.axvline(H_LAYER, color="gray", ls="--", lw=0.6)
        ax.set_title(f"{kind} push, r read at {'L' + str(R_LAYER) if where == 'r_layer' else f'L{late[0]}-{late[-1]} mean'}")
        ax.set_ylabel("fraction of gap closed")
axes[1, 0].set_xlabel("push layer"); axes[1, 1].set_xlabel("push layer"); axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

The same numbers as a table (late readout), plus the over-refusal check: how much the push raises `r` on harmless prompts. A push that closes the gap but also makes harmless prompts look harmful is not a fair fix.

In [ ]:
print(f"late readout L{late[0]}-{late[-1]}; gap closed by top-up h (random) | harmless r: base -> after top-up h")
for L in PUSH:
    cells = []
    for l in langs[1:]:
        hl = r_read(l, "harmless_val", f"topup_h@L{L}", "late").mean() - r_read(l, "harmless_val", "base", "late").mean()
        cells.append(f"{l} {closed(l, f'topup_h@L{L}', 'late'):5.2f} ({closed(l, f'topup_rand@L{L}', 'late'):5.2f}) +{hl:.2f}")
    print(f"L{L:2d}  " + " | ".join(cells))